# Mathematical Foundations

This notebook demonstrates how to encode a mathematical function and inspect its properties, as well as the basics of creating a function for code reuse.

In [ ]:
import numpy as np
from numpy import linalg as LA
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn import preprocessing


In [ ]:
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": "serif",
    "font.size": 6,
    "text.usetex": True,
    "pgf.rcfonts": False,
})

plt.rc('text.latex', preamble=r'\usepackage{amsmath}')

In [ ]:
# Get the colormap for consistent colors
cpal = plt.get_cmap('tab10')

# Globally assigned figure properties
FIG_WIDTH = 5.5129
FIG_HEIGHT = 2.0


# Gaussian Function

Recall the equation of the function itself:

$$y=\frac{1}{\sigma*\sqrt{\pi*2}}e^{-\frac{1}{2}*\left(\frac{(x-\mu)}{\sigma}^{2}\right)}$$

Our goal is to create a function that, given an $x$ value, will produce the corresponding Gaussian value via this formula.

In [ ]:
def gauss(x, mu=0.0, std=1.0):
    """Create a Gaussian response for a given mean, mu, and standard deviation, std."""
    factor = 1 / (std * np.sqrt(2*np.pi)  )

    # Note that the exponential is the only place that "x" appears
    exponential = np.exp( -(1/2) * ( (x - mu) / std)**2 )
    
    return  factor * exponential 

In [ ]:
# Create the "domain" -- the space upon which we want to model our Gaussian
x = np.linspace(-5, 5, 100)

In [ ]:
# The parameters of the Gaussian -- Mean and Standard Deviation
mu = 0
std = 1

# The result of the Gaussian function
y = gauss(x, mu, std)

In [ ]:
x

In [ ]:
y

In [ ]:
plt.plot(x,y)

## Plot the Function

In [ ]:
# Generate a figure and set of axes in which to plot stuff
fig, ax = plt.subplots()

# Plot the data in the axis
ax.plot(x, y, alpha=0.8, color='k', linewidth=1.0)

# Fill in the std curves
# std_range defines the different multiples of the standard deviation
# The range for each "blue" color goes from mu-std_range to mu+std_range
# We plot the "furthest-out" range first because it will get covered by the smaller, inner ranges

for idx, std_range in enumerate(np.array([4,3,2,1,0]) * std):
    std_domain = np.linspace(mu-std_range, mu+std_range, 100)
    
    # Color should be darkest in the middle, lightest at the edges
    # First fill should be multiplied by the highest fraction
    ax.fill_between(std_domain, gauss(std_domain, mu, std), facecolor=cpal(0), alpha=(idx+1)/8)
    ax.fill_between(std_domain, gauss(std_domain, mu, std), edgecolor='w', linewidth=0.5, facecolor=(0.0, 0.0, 0.0, 0.0))

# Add text boxes Below

## One for the equation in the left-hand side of the plot 
ax.text(-4, 0.31, r'$y=\frac{1}{\sigma*\sqrt{\pi*2}}e^{-\frac{1}{2}*\left(\frac{x-\mu}{\sigma}\right)}$',
            ha="left", va="center", size=12,
            bbox=dict(boxstyle="round,pad=0.3",
                      fc="white", ec="k", lw=1))

## One for the value of mu
# The position of the text is hand-crafted, the arrow should point to a specific spot on the gaussian
ax.annotate(r'$\mu=0.0$',
            xy=(0, gauss(0, mu, std)),
            xytext=(0.65, 0.8),    # fraction, fraction
            textcoords='figure fraction',
            arrowprops=dict(
                arrowstyle="->",
                connectionstyle='angle3,angleA=90,angleB=0',
                facecolor='black',
                
            ),
            horizontalalignment='left',
            verticalalignment='bottom')

## Two for the value of sigma
# We need two annotations here to create two lines pointing at the positive and negative std
ax.annotate(r'$\sigma=1.0$',
            xy=(1, gauss(1, mu, std)/2),
            xytext=(0.75, 0.65),    # fraction, fraction
            textcoords='figure fraction',
            arrowprops=dict(
                arrowstyle="->",
                connectionstyle='angle3,angleA=90,angleB=0',
                facecolor='black',
                
            ),
            horizontalalignment='left',
            verticalalignment='bottom')
ax.annotate(r'$\sigma=1.0$',
            xy=(-1, gauss(-1, mu, std)/2),
            xytext=(0.75, 0.65),    # fraction, fraction
            textcoords='figure fraction',
            arrowprops=dict(
                arrowstyle="->",
                connectionstyle='angle3,angleA=90,angleB=0',
                facecolor='black',
                
            ),
            horizontalalignment='left',
            verticalalignment='bottom',
           alpha=0.0)

# Set the properties of the plot, specifically the X and Y properties and the title
ax.set(xlim=(-5,5),
       ylim=(0,0.45),
       xlabel=r'$x$',
       ylabel=r'$y$',
       title=r'Gaussian Distribution for $\mathcal{N}(\mu=' + f'{mu}' + r', \sigma=' + f'{std}' + r')$)')

# Tweak Plot
ax.grid(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()

fig.set_size_inches(w=FIG_WIDTH, h=FIG_HEIGHT)
plt.show()

# Functions

What if we wanted to plot multiple Gaussians with different properties?

Let's create a *function* which will allow us to "box up" the code above, and use it multiple times without having to re-type everything.

Let's further make sure this function takes in *arguments*, meaning we can have the function create different Gaussians based on the input properties.

In [ ]:
def plot_gauss(xmin, xmax, mu, std, n = 100):
    """Plot a Gaussian function with a given range, mean, and standard deviation."""
    
    x = np.linspace(xmin, xmax, n)

    # Note here that our functions can call other functions, including ones we've built before
    y = gauss(x, mu, std)

    # Create the plot
    fig, ax = plt.subplots()
    ax.plot(x,y, alpha=0.8, color='k', linewidth=1.0)
    
    # Fill in the std curves
    for idx, std_range in enumerate(np.array([4,3,2,1,0])*std):
        std_domain = np.linspace(mu-std_range, mu+std_range, n)
        
        # Color should be darkest in the middle, lightest at the edges
        # First fill should be multiplied by the highest fraction
        ax.fill_between(std_domain, gauss(std_domain, mu, std), facecolor=cpal(0), alpha=(idx+1)/8)
        ax.fill_between(std_domain, gauss(std_domain, mu, std), edgecolor='w', linewidth=0.5, facecolor=(0.0, 0.0, 0.0, 0.0))

    # upper y limit is the value of gauss at the mean, plus a bit
    ax.set(xlim=(xmin,xmax),
           ylim=(0,gauss(mu, mu, std)+0.1),
           xlabel=r'$x$',
           ylabel=r'$y$',
           title=r'Gaussian Distribution for $\mathcal{N}(\mu=' + f'{mu}' + r', \sigma=' + f'{std}' + r')$)')
    
    # Tweak Plot
    ax.grid(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    plt.tight_layout()
    
    fig.set_size_inches(w=FIG_WIDTH, h=FIG_HEIGHT)
    plt.show()

Now we can simply call the function multiple times to plot multiple Gaussians:

In [ ]:
plot_gauss(5, 15, 10, 1)

We can do creative things like wrap the function in a loop to see how it changes with each parameter:

In [ ]:
for mu in [1, 2, 3]:
    for std in [1, 2, 3]:
        plot_gauss(-15, 15, mu, std)

# Similarity Matrix

To plot a similarity matrix, we first need to create one. We can approximate the similarity of two feature vectors by looking at the mathematical difference between them, and then taking the norm of the resulting vector, i.e.:

$$ \begin{align}
S(\mathbf{x}, \mathbf{y}) &= \left\lVert \mathbf{x} - \mathbf{y} \right\rVert &= \left\lVert \mathbf{z} \right\rVert \\
                          &= \left\lVert \begin{bmatrix}x_{1}\\x_{2}\\\vdots\\x_{d}\end{bmatrix} - 
                             \begin{bmatrix}y_{1}\\y_{2}\\\vdots\\y_{d}\end{bmatrix} \right\rVert &= 
                             \left\lVert \begin{bmatrix}z_{1}\\z_{2}\\\vdots\\z_{d}\end{bmatrix} \right\rVert\\
\end{align}$$

Let's do this with some of our real data.

In [ ]:
# Load up the breast cancer dataset to use
data_path = Path('../data/breast_cytology_features/wisconsin_breast_cancer_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Let's only look at the first eight patients and every other feature value
simdata = df.iloc[0:8, 2:32:2]

# Scale down the dataset
simdata_scaled = preprocessing.scale(simdata)

In [ ]:
# Set up the distance calculation
# The result should be a matrix with the same size as our samples, $n x n$ 
result = np.zeros((simdata_scaled.shape[0], simdata_scaled.shape[0]))

# For each location in the matrix, calculate the similarity
for i in range(simdata_scaled.shape[0]):
    for j in range(simdata_scaled.shape[0]):
        result[i,j] = LA.norm(simdata_scaled[i,:] - simdata_scaled[j,:])

In [ ]:
# Set up the plot
fig, ax = plt.subplots(figsize=(10,10))
ax.imshow(result)

for i in range(simdata_scaled.shape[0]):
    for j in range(simdata_scaled.shape[0]):
        text = ax.text(j, i, f'{result[i,j]:1.1f}',
                       ha="center", va="center", color="w")

fig.tight_layout()
#fig.set_size_inches(w=FIG_WIDTH, h=FIG_HEIGHT)

ax.set(title='Symmetric (Similarity) Matrix',
      xlabel='Subject Number',
      ylabel='Subject Number')

plt.show()

In [ ]:
# Set up the distance calculation
result = np.zeros((simdata_scaled.shape[0], simdata_scaled.shape[0]))
for i in range(simdata_scaled.shape[0]):
    for j in range(simdata_scaled.shape[0]):
        result[i,j] = np.sum(simdata_scaled[i,:] - simdata_scaled[j,:])

fig, ax = plt.subplots()
ax.imshow(result)

for i in range(simdata_scaled.shape[0]):
    for j in range(simdata_scaled.shape[0]):
        text = ax.text(j, i, f'{result[i,j]:1.1f}',
                       ha="center", va="center", color="w")

fig.tight_layout()
fig.set_size_inches(w=FIG_WIDTH, h=FIG_HEIGHT)

ax.set(title='Anti-Symmetric (Distance) Matrix',
      xlabel='Subject Number',
      ylabel='Subject Number')

# Save the plot
# plt.savefig(savepath / 'antisymmetric_matrix.pdf', bbox_inches='tight')
# plt.savefig(savepath / 'antisymmetric_matrix.pgf', bbox_inches='tight')
plt.show()

In [ ]:
# Vector Location in Space
# Create data plots
fig, ax = plt.subplots()

# These two lines do the work of plotting a histogram
ax.plot([0, 6], [0, 8])
txt = r'$\mathbf{x}=\begin{bmatrix}x_{1}\\x_{2}\end{bmatrix}\begin{bmatrix}x_{1}\\x_{2}\end{bmatrix}=\begin{bmatrix}6\\8\end{bmatrix}\begin{bmatrix}6\\8\end{bmatrix}$'
ax.annotate(txt,
            xy=(6, 8), xycoords='data',
            xytext=(3.5, 8), textcoords='data',
            bbox=dict(pad=0.5,boxstyle='round', facecolor='white', alpha=0.5),
            arrowprops=dict(arrowstyle="->",
                            connectionstyle="arc3,rad=-.2"))
ax.set(xlim=(0,7))
ax.set(ylim=(0,10))

# Annotate Plot
ax.set(xlabel=r'$x1$',
       ylabel=r'$x2$')

ax.set_title('\\textsc{' + r'Vector Representation in Cartesian Plane}', loc='left')

#ax.legend(frameon=True)
#ax.grid(linestyle=':')
ax.grid(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()

fig.set_size_inches(w=FIG_WIDTH, h=FIG_HEIGHT)

plt.show()